In [1]:
# 1. Installera inget - vi använder inbyggda bibliotek
# 2. Importera bibliotek
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# 3. Skapa realistisk IIoT-data
print("Genererar IIoT-sensor data...")

# Lista över fabriker med koordinater (verkliga städer)
fabriker = [
    {"name": "Stockholm Fabrik A", "city": "Stockholm", "country": "Sweden", "lat": 59.3293, "lon": 18.0686, "industry": "Automotive"},
    {"name": "Göteborg Fabrik B", "city": "Gothenburg", "country": "Sweden", "lat": 57.7089, "lon": 11.9746, "industry": "Shipping"},
    {"name": "Malmö Fabrik C", "city": "Malmo", "country": "Sweden", "lat": 55.6050, "lon": 13.0038, "industry": "Food Processing"},
    {"name": "Oslo Fabrik D", "city": "Oslo", "country": "Norway", "lat": 59.9139, "lon": 10.7522, "industry": "Oil & Gas"},
    {"name": "Bergen Fabrik E", "city": "Bergen", "country": "Norway", "lat": 60.3913, "lon": 5.3221, "industry": "Maritime"},
    {"name": "Helsingfors Fabrik F", "city": "Helsinki", "country": "Finland", "lat": 60.1699, "lon": 24.9384, "industry": "Electronics"},
    {"name": "Tammerfors Fabrik G", "city": "Tampere", "country": "Finland", "lat": 61.4981, "lon": 23.7610, "industry": "Paper & Pulp"},
]

# Sensortyper
sensor_typer = [
    {"type": "Temperature Sensor", "unit": "°C", "min": 15, "max": 35, "normal_range": (20, 30)},
    {"type": "Vibration Sensor", "unit": "mm/s", "min": 0, "max": 15, "normal_range": (0.5, 5)},
    {"type": "Pressure Sensor", "unit": "PSI", "min": 80, "max": 130, "normal_range": (95, 105)},
    {"type": "Energy Meter", "unit": "kWh", "min": 100, "max": 500, "normal_range": (200, 400)},
    {"type": "CO2 Sensor", "unit": "ppm", "min": 350, "max": 2000, "normal_range": (400, 800)},
]

# Generera data för 30 dagar, med mätningar var 3:e timme
start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 1, 30)
dates = pd.date_range(start_date, end_date, freq='3H')

data_rows = []

print(f"Skapar {len(fabriker) * len(sensor_typer) * len(dates)} datapunkter...")

for fabrik in fabriker:
    for sensor in sensor_typer:
        # Varje fabrik har 2-4 enheter av varje sensortyp
        num_devices = random.randint(2, 4)

        for device_id in range(1, num_devices + 1):
            device_name = f"{sensor['type']} #{device_id}"

            # Generera tidsdata
            for timestamp in dates:
                # Skapa normalt värde med slumpmässig variation
                if random.random() < 0.05:  # 5% chans för onormalt värde (anomali)
                    # Onormalt värde (simulerar problem)
                    if sensor["type"] == "Temperature Sensor":
                        value = random.uniform(45, 60)  # Överhettning
                    elif sensor["type"] == "Vibration Sensor":
                        value = random.uniform(10, 20)  # Kraftig vibration
                    elif sensor["type"] == "Pressure Sensor":
                        value = random.uniform(140, 160)  # Övertryck
                    elif sensor["type"] == "Energy Meter":
                        value = random.uniform(600, 800)  # Energispik
                    else:  # CO2
                        value = random.uniform(2500, 3500)  # Dålig ventilation
                    status = "WARNING"
                else:
                    # Normalt värde
                    mean_val = (sensor["min"] + sensor["max"]) / 2
                    std_val = (sensor["max"] - sensor["min"]) / 8
                    value = np.random.normal(mean_val, std_val)
                    value = max(sensor["min"], min(sensor["max"], value))  # Håll inom gränser
                    status = random.choice(["OK", "OK", "OK", "OK", "IDLE"])  # 80% OK, 20% IDLE

                # Skapa rad
                row = {
                    "Timestamp": timestamp,
                    "Date": timestamp.date(),
                    "Time": timestamp.time(),
                    "Factory": fabrik["name"],
                    "City": fabrik["city"],
                    "Country": fabrik["country"],
                    "Latitude": fabrik["lat"],
                    "Longitude": fabrik["lon"],
                    "Industry": fabrik["industry"],
                    "Device": device_name,
                    "SensorType": sensor["type"],
                    "Unit": sensor["unit"],
                    "Value": round(value, 2),
                    "Status": status,
                    "Hour": timestamp.hour,
                    "Weekday": timestamp.strftime("%A"),
                    "Month": timestamp.month,
                }
                data_rows.append(row)

# Skapa DataFrame
df = pd.DataFrame(data_rows)
print(f"Skapade {len(df)} rader med data!")

# Visa första raderna
print("\nFörsta 5 raderna i dataset:")
print(df.head())

# Enkel analys i Colab
print("\n--- ENKEL DATAANALYS I COLAB ---")
print(f"Antal mätningar: {len(df)}")
print(f"Antal fabriker: {df['Factory'].nunique()}")
print(f"Antal sensortyper: {df['SensorType'].nunique()}")
print(f"Tidsperiod: {df['Timestamp'].min()} till {df['Timestamp'].max()}")

print("\nStatus-fördelning:")
print(df['Status'].value_counts())

print("\nGenomsnittliga värden per sensortyp:")
print(df.groupby('SensorType')['Value'].mean().round(2))

# Hitta varningar (anomalier)
warnings = df[df['Status'] == 'WARNING']
print(f"\nAntal varningar: {len(warnings)}")
print("\nTopp 5 varningar:")
print(warnings[['Timestamp', 'Factory', 'SensorType', 'Value', 'Unit']].head())

# 4. Spara till CSV
csv_filename = "iiot_factory_data.csv"
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\n✅ Data sparad som: {csv_filename}")

# 5. Ladda ner filen till datorn
from google.colab import files
print("\nLaddar ner filen till din dator...")
files.download(csv_filename)
print("✅ Nedladdning klar! Filen finns nu i din Downloads-mapp.")

Genererar IIoT-sensor data...
Skapar 8155 datapunkter...


/tmp/ipykernel_9099/981238629.py:34: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(start_date, end_date, freq='3H')


Skapade 26096 rader med data!

Första 5 raderna i dataset:
            Timestamp        Date      Time             Factory       City  \
0 2024-01-01 00:00:00  2024-01-01  00:00:00  Stockholm Fabrik A  Stockholm   
1 2024-01-01 03:00:00  2024-01-01  03:00:00  Stockholm Fabrik A  Stockholm   
2 2024-01-01 06:00:00  2024-01-01  06:00:00  Stockholm Fabrik A  Stockholm   
3 2024-01-01 09:00:00  2024-01-01  09:00:00  Stockholm Fabrik A  Stockholm   
4 2024-01-01 12:00:00  2024-01-01  12:00:00  Stockholm Fabrik A  Stockholm   

  Country  Latitude  Longitude    Industry                 Device  \
0  Sweden   59.3293    18.0686  Automotive  Temperature Sensor #1   
1  Sweden   59.3293    18.0686  Automotive  Temperature Sensor #1   
2  Sweden   59.3293    18.0686  Automotive  Temperature Sensor #1   
3  Sweden   59.3293    18.0686  Automotive  Temperature Sensor #1   
4  Sweden   59.3293    18.0686  Automotive  Temperature Sensor #1   

           SensorType Unit  Value Status  Hour Weekday  M

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Nedladdning klar! Filen finns nu i din Downloads-mapp.
